In [11]:
import sys
import os
# Add parent directory to path to import tradingutils modules
sys.path.insert(0, os.path.abspath('..'))

from kalshi_utils.client_wrapper import *

kalshi_wrapped = KalshiWrapped()

nba_live_markets = kalshi_wrapped.GetALLNBAMarkets(status="open")
nhl_live_markets = kalshi_wrapped.GetAllNHLMarkets(status="open")
college_markets = kalshi_wrapped.GetALLNCAAMBMarkets(status="open")

pairs = kalshi_wrapped.GetMarketPairs(nba_live_markets)

Client Created


In [12]:
pairs

[(Market(ticker='KXNBAGAME-26JAN22MIAPOR-POR', event_ticker='KXNBAGAME-26JAN22MIAPOR', market_type='binary', title='Miami at Portland Winner?', subtitle='', yes_sub_title='Portland', no_sub_title='Portland', created_time=datetime.datetime(2026, 1, 20, 14, 3, 4, 696174, tzinfo=TzInfo(0)), open_time=datetime.datetime(2026, 1, 20, 19, 7, tzinfo=TzInfo(0)), close_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), expected_expiration_time=datetime.datetime(2026, 1, 23, 6, 0, tzinfo=TzInfo(0)), expiration_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), latest_expiration_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), settlement_timer_seconds=30, status='active', response_price_units='usd_cent', yes_bid=52, yes_bid_dollars='0.5200', yes_ask=55, yes_ask_dollars='0.5500', no_bid=45, no_bid_dollars='0.4500', no_ask=48, no_ask_dollars='0.4800', last_price=55, last_price_dollars='0.5500', volume=122, volume_24h=122, result='', can_close_early=True, open_interest=12

Error getting mid price: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '79', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:09:36 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "default-src 'none';", 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 ffd639fe55ad689097e5ac53454a5504.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'SFO53-P1', 'X-Amz-Cf-Id': '5x2QpGzfi2obL7H8nVddH3oT_2A71P1fmQLbqLQ6LukXUWZHJabNAw=='})
HTTP response body: {"error":{"code":"not_found","message":"not found","service":"query-exchange"}}

Error placing buy order: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '87', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:09:36 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "

In [10]:
import time
import threading
from datetime import datetime
from collections import deque
import uuid
import os
import logging

# ---------- Market Maker Bot with Stop/Take Limits ----------

class MarketMakerBot:
    def __init__(
        self,
        client,
        market,
        side: str = "yes",           # "yes" or "no"
        spread_bps: int = 50,        # spread in basis points (0.0050 = 50 bps)
        size_per_level: int = 1,     # contracts per level
        max_position: int = 10,      # max inventory (+ or -)
        stop_loss: float = -10.0,    # stop if P&L <= this ($)
        take_profit: float = 20.0,   # stop if P&L >= this ($)
        skew_on_inventory: bool = True,  # skew quotes based on inventory
        skew_bps_per_contract: int = 10, # how much to skew per contract
        check_interval_s: float = 2.0,   # how often to update quotes
        logger: logging.Logger = None,
    ):
        self.client = client
        self.market = market
        self.ticker = market.model_dump()['ticker']
        self.side = side
        self.spread_bps = spread_bps / 10000.0  # convert to decimal
        self.size_per_level = size_per_level
        self.max_position = max_position
        self.stop_loss = stop_loss
        self.take_profit = take_profit
        self.skew_on_inventory = skew_on_inventory
        self.skew_bps_per_contract = skew_bps_per_contract / 10000.0
        self.check_interval_s = check_interval_s
        
        # Logging setup
        if logger is None:
            log_dir = "logs"
            os.makedirs(log_dir, exist_ok=True)
            session_id = str(uuid.uuid4())[:8]
            date_str = datetime.now().strftime("%Y-%m-%d")
            log_filename = f"mm_{date_str}_{session_id}.log"
            log_path = os.path.join(log_dir, log_filename)
            
            self.logger = logging.getLogger(f"mm_{session_id}")
            self.logger.setLevel(logging.DEBUG)
            self.logger.handlers = []
            
            fh = logging.FileHandler(log_path)
            fh.setLevel(logging.DEBUG)
            fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
            self.logger.addHandler(fh)
            
            ch = logging.StreamHandler()
            ch.setLevel(logging.INFO)
            ch.setFormatter(logging.Formatter('%(message)s'))
            self.logger.addHandler(ch)
            
            self.logger.info(f"=== Market Maker Bot started: {log_path} ===")
        else:
            self.logger = logger
        
        # State tracking
        self.position = 0           # current inventory (net contracts)
        self.realized_pnl = 0.0     # locked in P&L
        self.avg_entry_price = 0.0  # average price of current position
        self.total_filled = 0       # total contracts traded
        self.trade_history = []
        self.active_orders = {}     # order_id -> order_info
        
        self.stop_evt = threading.Event()
        self.running = False
        self.thread = None
        self.logger.info(f"  Ticker: {self.ticker}, Side: {side}")
        self.logger.info(f"  Spread: {spread_bps} bps, Size: {size_per_level}")
        self.logger.info(f"  Max Position: ±{max_position}")
        self.logger.info(f"  Stop Loss: ${stop_loss}, Take Profit: ${take_profit}")
        self.logger.info(f"  Skew on inventory: {skew_on_inventory}")
    
    def start(self):
        """Start the market making bot"""
        if self.running:
            self.logger.warning("Bot already running")
            return self
        
        self.running = True
        self.stop_evt.clear()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        self.logger.info("✓ Market maker bot started")
        return self
    
    def stop(self):
        """Stop the bot and cancel all orders"""
        if not self.running:
            return
        
        self.logger.info("Stopping market maker bot...")
        self.stop_evt.set()
        self._cancel_all_orders()
        
        if self.thread:
            self.thread.join(timeout=5.0)
        
        self.running = False
        self.logger.info("✓ Market maker bot stopped")
        self._print_summary()
    
    def _get_mid_price(self):
        """Get current mid price of the market"""
            # Refresh market data
            market_response = self.client.get_market(self.ticker)
            data = market_response.market.model_dump()
            
            if self.side == "yes":
                bid = float(data.get("yes_bid_dollars", 0))
                ask = float(data.get("yes_ask_dollars", 1))
            else:
                bid = float(data.get("no_bid_dollars", 0))
                ask = float(data.get("no_ask_dollars", 1))
            
            if bid > 0 and ask > 0 and ask > bid:
                mid = (bid + ask) / 2.0
            else:
                # Fallback to last price or 0.5
                mid = 0.5
            
            return mid, bid, ask
        except Exception as e:
            self.logger.error(f"Error getting mid price: {e}")
            return 0.5, 0.0, 1.0
    
    def _calculate_quotes(self):
        """Calculate bid/ask quotes based on mid price and inventory"""
        mid, market_bid, market_ask = self._get_mid_price()
        
        # Base quotes: mid ± spread/2
        base_bid = mid - self.spread_bps / 2.0
        base_ask = mid + self.spread_bps / 2.0
        
        # Apply inventory skew if enabled
        if self.skew_on_inventory and self.position != 0:
            # If long (positive position), lower both quotes to encourage selling
            # If short (negative position), raise both quotes to encourage buying
            skew = self.position * self.skew_bps_per_contract
            base_bid -= skew
            base_ask -= skew
        
        # Clamp to valid range [0.01, 0.99]
        bid = max(0.01, min(0.99, base_bid))
        ask = max(0.01, min(0.99, base_ask))
        
        # Ensure ask > bid
        if ask <= bid:
            mid_point = (bid + ask) / 2.0
            bid = mid_point - 0.005
            ask = mid_point + 0.005
        
        return bid, ask, mid
    
    def _should_quote(self):
        """Check if we should place quotes given current position"""
        # Don't buy if at max long position
        can_buy = self.position < self.max_position
        # Don't sell if at max short position
        can_sell = self.position > -self.max_position
        
        return can_buy, can_sell
    
    def _place_order(self, action: str, price: float, quantity: int):
        """Place a limit order"""
        try:
            price_cents = int(round(price * 100))
            
            order_params = {
                "ticker": self.ticker,
                "action": action,
                "side": self.side,
                "count": quantity,
                "type": "limit",
            }
            
            if self.side == "yes":
                order_params["yes_price"] = price_cents
            else:
                order_params["no_price"] = price_cents
            response = self.client.create_order(**order_params)
            response = self.client.create_order(**order_params)
            order_id = response.order.order_id if hasattr(response, 'order') else None
            order_id = response.order.order_id if hasattr(response, 'order') else None
            
            if order_id:
                self.active_orders[order_id] = {
                    "action": action,
                    "price": price,
                    "quantity": quantity,
                    "time": time.time()
                }
                self.logger.debug(f"Placed {action} order: {quantity} @ ${price:.3f} (id={order_id})")
            
            return order_id
        except Exception as e:
            self.logger.error(f"Error placing {action} order: {e}")
            return None
    
    def _cancel_all_orders(self):
        """Cancel all active orders"""
        if not self.active_orders:
            return
        self.logger.info(f"Canceling {len(self.active_orders)} active orders...")
        self.logger.info(f"Canceling {len(self.active_orders)} active orders...")
        
        for order_id in list(self.active_orders.keys()):
            try:
                self.client.cancel_order(order_id=order_id)
                self.logger.debug(f"Canceled order {order_id}")
            except Exception as e:
                self.logger.debug(f"Error canceling order {order_id}: {e}")
        
        self.active_orders.clear()
    
    def _check_fills(self):
        """Check for filled orders and update position"""
        if not self.active_orders:
            return
        
        for order_id in list(self.active_orders.keys()):
            try:
                response = self.client.get_order(order_id=order_id)
                status = response.order.status if hasattr(response, 'order') else None
                
                if status == "filled":
                    order_info = self.active_orders[order_id]
                    self._handle_fill(order_info)
                elif status in ["canceled", "expired"]:
                elif status in ["canceled", "expired"]:
                    del self.active_orders[order_id]
            except Exception as e:
                self.logger.debug(f"Error checking order {order_id}: {e}")
    
    def _handle_fill(self, order_info):
        """Handle a filled order and update P&L"""
        action = order_info["action"]
        price = order_info["price"]
        quantity = order_info["quantity"]
        
        # Update position
        if action == "buy":
            # Buying increases position
            new_position = self.position + quantity
            
            # Update average entry price
            if self.position >= 0:
                # Adding to long or flipping from flat
                total_cost = self.position * self.avg_entry_price + quantity * price
                self.avg_entry_price = total_cost / new_position if new_position != 0 else price
            else:
                # Reducing short position - realize P&L
                contracts_covered = min(quantity, abs(self.position))
                pnl = contracts_covered * (self.avg_entry_price - price)
                self.realized_pnl += pnl
                
                remaining = quantity - contracts_covered
                if remaining > 0:
                    # Flipped to long
                    self.avg_entry_price = price
            self.position = new_position
            self.position = new_position
            
        else:  # action == "sell"
            # Selling decreases position
            new_position = self.position - quantity
            
            # Update average entry price
            if self.position <= 0:
                # Adding to short or flipping from flat
                total_cost = abs(self.position) * self.avg_entry_price + quantity * price
                self.avg_entry_price = total_cost / abs(new_position) if new_position != 0 else price
            else:
                # Reducing long position - realize P&L
                contracts_sold = min(quantity, self.position)
                pnl = contracts_sold * (price - self.avg_entry_price)
                self.realized_pnl += pnl
                
                remaining = quantity - contracts_sold
                if remaining > 0:
                    # Flipped to short
                    self.avg_entry_price = price
            
            self.position = new_position
        
        self.total_filled += quantity
        self.trade_history.append({
            "time": time.time(),
            "action": action,
            "price": price,
            "quantity": quantity,
            "position_after": self.position
        })
        
        ts = datetime.now().strftime("%H:%M:%S")
        unrealized = self._calculate_unrealized_pnl()
        total_pnl = self.realized_pnl + unrealized
        
        msg = (f"[{ts}] FILL: {action.upper()} {quantity} @ ${price:.3f} | "
               f"Pos: {self.position:+d} | P&L: ${total_pnl:.2f} "
               f"(realized: ${self.realized_pnl:.2f}, unrealized: ${unrealized:.2f})")
        print(msg)
        self.logger.info(msg)
    
    def _calculate_unrealized_pnl(self):
        """Calculate unrealized P&L on current position"""
        if self.position == 0:
            return 0.0
        mid, _, _ = self._get_mid_price()
        mid, _, _ = self._get_mid_price()
        
        if self.position > 0:
            # Long position: unrealized = position * (current_price - avg_entry)
            unrealized = self.position * (mid - self.avg_entry_price)
        else:
            # Short position: unrealized = abs(position) * (avg_entry - current_price)
            unrealized = abs(self.position) * (self.avg_entry_price - mid)
        
        return unrealized
    
    def get_total_pnl(self):
        """Get total P&L (realized + unrealized)"""
        return self.realized_pnl + self._calculate_unrealized_pnl()
    
    def _check_risk_limits(self):
        """Check if stop loss or take profit hit"""
        total_pnl = self.get_total_pnl()
        
        if total_pnl <= self.stop_loss:
            msg = f"⚠ STOP LOSS HIT: P&L ${total_pnl:.2f} <= ${self.stop_loss:.2f}"
            print(msg)
            self.logger.warning(msg)
            return True
        
        if total_pnl >= self.take_profit:
            msg = f"✓ TAKE PROFIT HIT: P&L ${total_pnl:.2f} >= ${self.take_profit:.2f}"
            print(msg)
            self.logger.info(msg)
            return True
        
        return False
    
    def _run(self):
        """Main market making loop"""
        while not self.stop_evt.is_set():
            try:
                # Check for fills first
                self._check_fills()
                
                # Check risk limits
                if self._check_risk_limits():
                    self.stop()
                    break
                
                # Cancel existing orders
                self._cancel_all_orders()
                
                # Calculate new quotes
                bid, ask, mid = self._calculate_quotes()
                can_buy, can_sell = self._should_quote()
                
                ts = datetime.now().strftime("%H:%M:%S")
                total_pnl = self.get_total_pnl()
                
                # Place new orders
                if can_buy:
                    self._place_order("buy", bid, self.size_per_level)
                
                if can_sell:
                    self._place_order("sell", ask, self.size_per_level)
                
                self.logger.debug(
                    f"[{ts}] Quotes: bid=${bid:.3f} ask=${ask:.3f} mid=${mid:.3f} | "
                    f"Pos: {self.position:+d}/{self.max_position} | P&L: ${total_pnl:.2f}"
                )
                
                # Wait before next update
                time.sleep(self.check_interval_s)
                
            except Exception as e:
                self.logger.error(f"Error in market maker loop: {e}", exc_info=True)
                time.sleep(self.check_interval_s)
    
    def _print_summary(self):
        """Print trading summary"""
        print("\n" + "="*60)
        print("MARKET MAKER SUMMARY")
        print("="*60)
        print(f"Ticker: {self.ticker} ({self.side})")
        print(f"Total Contracts Filled: {self.total_filled}")
        print(f"Final Position: {self.position:+d}")
        print(f"Realized P&L: ${self.realized_pnl:.2f}")
        print(f"Unrealized P&L: ${self._calculate_unrealized_pnl():.2f}")
        print(f"Total P&L: ${self.get_total_pnl():.2f}")
        print("="*60 + "\n")

# pairs = kalshi_wrapped.GetMarketPairs(college_markets)
# selected_market = pairs[0][0]  # Select a market from pairs
# 
# bot = MarketMakerBot(
#     client=kalshi_wrapped.GetClient(),
#     market=selected_market,     # Pass the market object
#     side="yes",
#     spread_bps=50,          # 0.5% spread
#     size_per_level=1,       # 1 contract per level
#     max_position=10,        # max ±10 contracts
#     stop_loss=-10.0,        # stop if lose $10
#     take_profit=20.0,       # stop if profit $20
#     skew_on_inventory=True, # skew quotes based on position
#     skew_bps_per_contract=10,  # 0.1% skew per contract
#     check_interval_s=2.0,   # update every 2 seconds
# ).start()
# To check status:
# To check status:
# print(f"Position: {bot.position}, P&L: ${bot.get_total_pnl():.2f}")

#
# To stop:
# bot.stop()
# To stop:# bot.stop()

# To stop:
# bot.stop()# bot.stop()

IndentationError: unexpected indent (1610626638.py, line 116)

Error getting mid price: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '79', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:09:26 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "default-src 'none';", 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 ffd639fe55ad689097e5ac53454a5504.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'SFO53-P1', 'X-Amz-Cf-Id': 'z0xp3kzLDBFnmG7Jnx-VHv4j2c40QtH9b1cbL0JQUNyNtGBIQHA28g==', 'Age': '7'})
HTTP response body: {"error":{"code":"not_found","message":"not found","service":"query-exchange"}}

Error placing buy order: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '87', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:09:33 GMT', 'x-content-type-options': 'nosniff', 'content-securit

## Market Maker Bot Configuration

Select a market and configure the bot parameters below:

**Key Parameters:**
- `spread_bps`: Spread in basis points (50 bps = 0.5% = $0.005)
- `size_per_level`: Number of contracts to quote on each side
- `max_position`: Maximum inventory (±contracts). Bot won't buy if at +max, won't sell if at -max
- `stop_loss`: Stop trading if P&L drops to this level (in dollars)
- `take_profit`: Stop trading if P&L reaches this level (in dollars)
- `skew_on_inventory`: If True, skew quotes when inventory builds up
  - Long inventory → lower quotes to encourage selling
  - Short inventory → raise quotes to encourage buying
- `skew_bps_per_contract`: How much to adjust quotes per contract of inventory
- `check_interval_s`: How often to update quotes (seconds)

In [5]:
# Step 1: Select a market from the available pairs
# Example: Use the first college basketball market pair
if college_markets:
    pairs = kalshi_wrapped.GetMarketPairs(college_markets)
    if pairs:
        # Show first 3 pairs
        print("Available market pairs:")
        for i, pair in enumerate(pairs[:3]):
            m1, m2 = pair
            print(f"\n{i+1}. {m1.model_dump()['ticker']}")
            print(f"   {m1.model_dump()['title']}")
            print(f"   {m2.model_dump()['ticker']}")
            print(f"   {m2.model_dump()['title']}")
        
        # Select the first market's first side
        selected_market = pairs[0][0]
        print(f"\n✓ Selected market: {selected_market.model_dump()['ticker']}")
        print(f"   Title: {selected_market.model_dump()['title']}")
else:
    print("No college markets available. Try NBA or NHL markets.")

Available market pairs:

1. KXNCAAMBGAME-26JAN23CSBHAW-HAW
   Cal State Bakersfield at Hawai'i Winner?
   KXNCAAMBGAME-26JAN23CSBHAW-CSB
   Cal State Bakersfield at Hawai'i Winner?

2. KXNCAAMBGAME-26JAN22UCIUCRV-UCRV
   UC Irvine at UC Riverside Winner?
   KXNCAAMBGAME-26JAN22UCIUCRV-UCI
   UC Irvine at UC Riverside Winner?

3. KXNCAAMBGAME-26JAN22LBSUCSF-LBSU
   Long Beach St. at Cal State Fullerton Winner?
   KXNCAAMBGAME-26JAN22LBSUCSF-CSF
   Long Beach St. at Cal State Fullerton Winner?

✓ Selected market: KXNCAAMBGAME-26JAN23CSBHAW-HAW


In [7]:
# Manual market selection (if needed)

# Find a specific market from the available marketsprint(f"Selected: {selected_market.model_dump()['ticker']}")
selected_market = nba_live_markets[0] if nba_live_markets else college_markets[0]

In [8]:
# Step 2: Create and start the market maker bot
bot = MarketMakerBot(
    client=kalshi_wrapped.GetClient(),
    market=selected_market,       # Pass the market object
    side="yes",                   # "yes" or "no"
    spread_bps=50,                # 0.5% spread (50 basis points)
    size_per_level=1,             # 1 contract per level
    max_position=5,               # max ±5 contracts inventory
    stop_loss=-5.0,               # stop if P&L drops to -$5
    take_profit=10.0,             # stop if P&L reaches +$10
    skew_on_inventory=True,       # adjust quotes based on inventory
    skew_bps_per_contract=10,     # 0.1% adjustment per contract
    check_interval_s=3.0,         # update quotes every 3 seconds
).start()

print(f"\n✓ Market maker bot started on {bot.ticker}")
print(f"  Monitoring P&L. Will stop if P&L <= -$5 or >= +$10")
print(f"  Run bot.stop() to manually stop the bot")

=== Market Maker Bot started: logs/mm_KXNBAGAME-26JAN20LALDEN_2026-01-20_c82ae563.log ===
MarketMakerBot initialized:
  Ticker: KXNBAGAME-26JAN20LALDEN, Side: yes
  Spread: 50 bps, Size: 1
  Max Position: ±5
  Stop Loss: $-5.0, Take Profit: $10.0
  Skew on inventory: True
✓ Market maker bot started



✓ Market maker bot started on KXNBAGAME-26JAN20LALDEN
  Monitoring P&L. Will stop if P&L <= -$5 or >= +$10
  Run bot.stop() to manually stop the bot


Error getting mid price: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '79', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:07:34 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "default-src 'none';", 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 ffd639fe55ad689097e5ac53454a5504.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'SFO53-P1', 'X-Amz-Cf-Id': 'ibFQjA6sftexZuM_Mg4q5ft1Y0UDsdxLLn0MEKv0u8smZ_ITh9nFwQ=='})
HTTP response body: {"error":{"code":"not_found","message":"not found","service":"query-exchange"}}

Error placing buy order: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '87', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 05:07:34 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "

In [ ]:
# Step 3: Monitor the bot status
print(f"Position: {bot.position:+d} contracts")
print(f"Realized P&L: ${bot.realized_pnl:.2f}")
print(f"Unrealized P&L: ${bot._calculate_unrealized_pnl():.2f}")
print(f"Total P&L: ${bot.get_total_pnl():.2f}")
print(f"Total Filled: {bot.total_filled} contracts")
print(f"Active Orders: {len(bot.active_orders)}")

In [ ]:
# Step 4: View trade history
if bot.trade_history:
    import pandas as pd
    df = pd.DataFrame(bot.trade_history)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    print("\nTrade History:")
    print(df.to_string(index=False))
else:
    print("No trades executed yet")

In [ ]:
# Step 5: Stop the bot manually (if needed)
bot.stop()
# The bot will automatically stop when stop_loss or take_profit is hit